# Program 05: Multiple Linear Regression

In [4]:
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split


# Find the data file from either the repository root or this notebook's folder.
data_candidates = []
for base_path in [Path.cwd(), *Path.cwd().parents]:
    data_candidates.extend(
        [
            base_path / "data" / "startup_data.csv",
            base_path / "Machine-Learning-Lab" / "data" / "startup_data.csv",
        ]
    )

DATA_PATH = next((path for path in data_candidates if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find startup_data.csv. Expected it at Machine-Learning-Lab/data/startup_data.csv."
    )

try:
    data = pd.read_csv(DATA_PATH)
except pd.errors.EmptyDataError as error:
    raise ValueError(
        f"{DATA_PATH} is empty. Add the startup records before running the model."
    ) from error

required_columns = {
    "R&D Spend",
    "Administration",
    "Marketing Spend",
    "State",
    "Profit",
}
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

features = ["R&D Spend", "Administration", "Marketing Spend", "State"]
X = pd.get_dummies(data[features], columns=["State"], drop_first=True, dtype=float)
y = data["Profit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=0
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Rows loaded: {len(data)}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred):,.2f}")

# Keep the same encoded columns used during training for a new prediction.
new_startup = pd.DataFrame(
    [{
        "R&D Spend": 130000,
        "Administration": 120000,
        "Marketing Spend": 300000,
        "State": "New York",
    }]
)
new_startup_encoded = pd.get_dummies(
    new_startup, columns=["State"], drop_first=True, dtype=float
).reindex(columns=X.columns, fill_value=0)

predicted_profit = model.predict(new_startup_encoded)[0]
print(f"Predicted Profit: ${predicted_profit:,.2f}")

Rows loaded: 5
R2 Score: -1496.9689
Mean Squared Error: 549,599,890.49
Predicted Profit: $203,465.13
